## Import Requirements

In [ ]:
import pandas as pd
import re
import numpy as np
import math

import contractions
import fasttext

from sentence_transformers import SentenceTransformer
from bertopic import BERTopic
from umap import UMAP
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel

C:\Users\kylie\AppData\Roaming\Python\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#### Preprocessing Step

In [ ]:
# Load csv file into songs_df dataframe
songs_df = pd.read_csv("../../Tmdata/DATA/songs_with_vader.csv")

print("Songs shape:", songs_df.shape)
print("\nSongs columns:", songs_df.columns.tolist())
songs_df.head()

Songs shape: (520589, 23)

Songs columns: ['id', 'name', 'artists', 'danceability', 'energy', 'loudness', 'speechiness', 'valence', 'tempo', 'lyrics', 'year', 'genre', 'popularity', 'total_artist_followers', 'avg_artist_popularity', 'niche_genres', 'lyrics_clean', 'lyrics_normalized', 'language', 'non_english_ratio', 'word_count', 'vader_compound', 'vader_sentiment']


,id,name,artists,danceability,energy,loudness,speechiness,valence,tempo,lyrics,...,total_artist_followers,avg_artist_popularity,niche_genres,lyrics_clean,lyrics_normalized,language,non_english_ratio,word_count,vader_compound,vader_sentiment
0,0Prct5TDjAnEgIqbxcldY9,!,"[""HELLYEAH""]",0.415,0.605,-11.157,0.0575,0.193,100.059,"He said he came from Jamaica,\nhe owned a coup...",...,769490,52.0,"[""groove metal"", ""metal""]","He said he came from Jamaica,\nhe owned a coup...","he said he came from jamaica, he owned a coupl...",en,0.229474,475,0.9574,Positive
1,2ASl4wirkeYm3OWZxXKYuq,!!,"[""Yxngxr1""]",0.788,0.648,-9.135,0.3150,0.287,79.998,"Fuck the bitch, now she running with my kids\n...",...,143628,45.0,[],"Fuck the bitch, now she running with my kids\n...","fuck the bitch, now she running with my kids a...",en,0.175781,256,0.9982,Positive
2,5tA3ImW310llKo8EMBj2Ga,!!Noble Stabbings!!,"[""Dillinger Four""]",0.171,0.957,-5.749,0.1490,0.349,175.317,You like to stand on the other side\nPoint and...,...,36619,35.0,"[""melodic hardcore"", ""pop punk"", ""punk"", ""skat...",You like to stand on the other side\nPoint and...,you like to stand on the other side point and ...,en,0.127962,211,-0.9868,Negative
3,1xBFhv5faebv3mmwxx7DnS,!Lost!,"[""Ril\u00e8s""]",0.729,0.552,-8.562,0.0650,0.380,86.103,I would like to give you all my time\nI would ...,...,929303,63.0,"[""french rap""]",I would like to give you all my time\nI would ...,i would like to give you all my time i would l...,en,0.111732,358,-0.2434,Negative
4,0gNNToCW3qjabgTyBSjt3H,!Que Vida! - Mono Version,"[""Love""]",0.600,0.540,-11.803,0.0328,0.547,125.898,With pictures and words\nIs this communicating...,...,273598,46.0,"[""acid rock"", ""baroque pop"", ""proto-punk"", ""ps...",With pictures and words\nIs this communicating...,with pictures and words is this communicating?...,en,0.170732,123,-0.8528,Negative


In [5]:
# Inspect structure of the lyrics column
songs_df['lyrics_normalized'][0]

'he said he came from jamaica, he owned a couple acres a couple fake visas \'cause he never got his papers gave up on love fucking with them heart breakers but he was getting money with the movers and the shakers he was mixed with a couple things, bald like a couple rings bricks in the condo and grams to sing-sing left arm, baby mother tatted 5-year bid up north when they ratted anyway i felt him, helped him, put him on lock, seat-belt him took him out to belgium, welcome bitches this pretty, that\'s seldom this box better than the box he was held in i\'m momma in that order, i call him daddy like daughters he like it when i get drunk, but i like it when he be sober that\'s top of the toppa, i never fuck with beginners i let him play with my pussy then lick it off of his fingers, i\'m in the zone they holler at me, but it\'s you, you, this is not high school me and my crew, we can slide through give it to you whenever you want, whip it whenever you want baby it\'s yours anywhere, every

In [6]:
# Preprocessing function
def clean_lyrics(text):
    if not isinstance(text, str):
        return ""
    
    # Replace literal '\n' with actual newlines
    text = text.replace('\\n', '\n')
    
    # Split into lines
    lines = text.split('\n')
    
    cleaned_lines = []
    previous_line = None
    
    for line in lines:
        line = line.strip()
        if line != previous_line and line != '':
            # Expand contractions
            line = contractions.fix(line)
            cleaned_lines.append(line)
            previous_line = line
    
    # Join lines back with a single newline
    return '\n'.join(cleaned_lines)

# Apply to your DataFrame
songs_df['final_lyrics_cleaned'] = songs_df['lyrics_normalized'].apply(clean_lyrics)

In [7]:
# Check first song
print(songs_df['final_lyrics_cleaned'].iloc[0])

he said he came from jamaica, he owned a couple acres a couple fake visas because he never got his papers gave up on love fucking with them heart breakers but he was getting money with the movers and the shakers he was mixed with a couple things, bald like a couple rings bricks in the condo and grams to sing-sing left arm, baby mother tatted 5-year bid up north when they ratted anyway i felt him, helped him, put him on lock, seat-belt him took him out to belgium, welcome bitches this pretty, that is seldom this box better than the box he was held in i am momma in that order, i call him daddy like daughters he like it when i get drunk, but i like it when he be sober that is top of the toppa, i never fuck with beginners i let him play with my pussy then lick it off of his fingers, i am in the zone they holler at me, but it is you, you, this is not high school me and my crew, we can slide through give it to you whenever you want, whip it whenever you want baby it is yours anywhere, everyw

In [8]:
# Load the FastText language identification model
model = fasttext.load_model("lid.176.bin")  # make sure this path points to where you saved lid.176.bin

# Function to check if the song is English
def is_english_song(text):
    # Replace \n with space to make it one line
    text_one_line = text.replace('\n', ' ')
    lang, prob = model.predict(text_one_line)
    return lang[0] == "__label__en" and prob[0] > 0.9


print(f"Number of English songs before filtering: {len(songs_df)}")
# Apply function and filter in-place
songs_df = songs_df[songs_df['final_lyrics_cleaned'].apply(is_english_song)].reset_index(drop=True)

print(f"Number of English songs after filtering: {len(songs_df)}")

Number of English songs before filtering: 520589
Number of English songs after filtering: 453384


In [9]:
# Check result
print(songs_df['final_lyrics_cleaned'].iloc[0])

he said he came from jamaica, he owned a couple acres a couple fake visas because he never got his papers gave up on love fucking with them heart breakers but he was getting money with the movers and the shakers he was mixed with a couple things, bald like a couple rings bricks in the condo and grams to sing-sing left arm, baby mother tatted 5-year bid up north when they ratted anyway i felt him, helped him, put him on lock, seat-belt him took him out to belgium, welcome bitches this pretty, that is seldom this box better than the box he was held in i am momma in that order, i call him daddy like daughters he like it when i get drunk, but i like it when he be sober that is top of the toppa, i never fuck with beginners i let him play with my pussy then lick it off of his fingers, i am in the zone they holler at me, but it is you, you, this is not high school me and my crew, we can slide through give it to you whenever you want, whip it whenever you want baby it is yours anywhere, everyw

In [10]:
lyrics_stopwords = {
    'chorus', 'verse','hook', 'bridge', 'repeat', 'outro', 'intro', 'prechorus', 'postchorus', 'interlude'
}

def clean_lyrics_keep_lines(text):
    lines = text.split('\n')  # keep each line
    cleaned_lines = []
    
    for line in lines:
        line = line.lower().strip()
        # remove punctuation/numbers, keep spaces
        line = re.sub(r'[^a-z\s]', ' ', line)
        # split into tokens, remove stopwords and single letters
        tokens = [t for t in line.split() if t and t not in lyrics_stopwords and len(t) > 1]
        if tokens:  # only add non-empty lines
            cleaned_lines.append(' '.join(tokens))

    # final check to remove any leftover empty lines
    cleaned_lines = [line for line in cleaned_lines if line.strip()]
    return '\n'.join(cleaned_lines)

songs_df['final_lyrics_cleaned'] = songs_df['final_lyrics_cleaned'].apply(clean_lyrics_keep_lines)

In [11]:
# Check result
print(songs_df['final_lyrics_cleaned'].iloc[0])

he said he came from jamaica he owned couple acres couple fake visas because he never got his papers gave up on love fucking with them heart breakers but he was getting money with the movers and the shakers he was mixed with couple things bald like couple rings bricks in the condo and grams to sing sing left arm baby mother tatted year bid up north when they ratted anyway felt him helped him put him on lock seat belt him took him out to belgium welcome bitches this pretty that is seldom this box better than the box he was held in am momma in that order call him daddy like daughters he like it when get drunk but like it when he be sober that is top of the toppa never fuck with beginners let him play with my pussy then lick it off of his fingers am in the zone they holler at me but it is you you this is not high school me and my crew we can slide through give it to you whenever you want whip it whenever you want baby it is yours anywhere everywhere baby it is your world is not it alright

#### Applying the Model

In [12]:
documents = songs_df['final_lyrics_cleaned'].tolist()
print(f"Number of documents: {len(documents)}")
print(f"Sample document length: {len(documents[0])} characters")

Number of documents: 453384
Sample document length: 2215 characters


In [13]:
sample_df = songs_df.sample(20000, random_state=42).reset_index(drop=True)
sample_documents = sample_df['final_lyrics_cleaned'].tolist()

print(f"Number of sampled documents: {len(sample_documents)}")
print(f"Sample document length: {len(sample_documents[0])} characters")

Number of sampled documents: 20000
Sample document length: 1189 characters


In [14]:
def chunk_lyrics(documents, min_words=12):
    """
    Splits a list of cleaned song strings into fixed-size word chunks.

    Parameters
    ----------
    documents : list of str   — one cleaned song per element
    min_words : int           — minimum words per chunk

    Returns
    -------
    chunks   : list of str   — all text chunks
    song_map : list of int   — index into `documents` for each chunk

    Works for both training (full list) and demo (single song):
        training : chunk_lyrics(sample_documents)
        demo     : chunk_lyrics([preprocess_single_song(raw_lyrics)])
    """
    chunks = []
    song_map = []
    for idx, song in enumerate(documents):
        lines = song.split('\n')
        buffer = []
        for line in lines:
            line = line.strip()
            if not line:
                continue
            buffer.extend(line.split())
            while len(buffer) >= min_words:
                chunks.append(' '.join(buffer[:min_words]))
                song_map.append(idx)
                buffer = buffer[min_words:]
        if buffer:
            if chunks and song_map[-1] == idx:
                chunks[-1] += ' ' + ' '.join(buffer)
            else:
                chunks.append(' '.join(buffer))
                song_map.append(idx)
    return chunks, song_map


sample_line_documents, song_map = chunk_lyrics(sample_documents, min_words=12)

print(f"Total chunks: {len(sample_line_documents)}")
print(f"Sample chunk: {sample_line_documents[0]}")

Total chunks: 395773
Sample chunk: little sleepy boy do you know what time it is well the


In [15]:
# Inspect chunks for the first song
first_song_index = 0
first_song_chunks = [
    chunk for chunk, idx in zip(sample_line_documents, song_map)
    if idx == first_song_index
]
for i, chunk in enumerate(first_song_chunks, start=1):
    print(f"Chunk {i}: {chunk}")

Chunk 1: little sleepy boy do you know what time it is well the
Chunk 2: hour of your bedtime long been past and though know you are
Chunk 3: fighting it can tell when you rub your eyes you are fading
Chunk 4: fast fading fast will not you run come see st judy comet
Chunk 5: roll across the skies and leave spray of diamonds in its wake
Chunk 6: long to see st judy comet sparkle in your eyes when you
Chunk 7: awake little boy will not you lay your body down little boy
Chunk 8: will not you close your weary eyes is not nothing flashing but
Chunk 9: the fireflies well sang it once then sang it twice am going
Chunk 10: to sing it three times more am going to stay til your
Chunk 11: resistance is overcome because if cannot sing my boy to sleep well
Chunk 12: it makes your famous daddy look so dumb will not you run
Chunk 13: come see st judy comet roll across the skies and leave spray
Chunk 14: of diamonds in its wake long to see st judy comet sparkle
Chunk 15: in your eyes when you awake li

In [16]:
# 1. Initialise embedding model
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# 2. Compute embeddings in batches
batch_size = 256
all_embeddings = []
num_batches = math.ceil(len(sample_line_documents) / batch_size)

for i in range(num_batches):
    batch_lines = sample_line_documents[i*batch_size : (i+1)*batch_size]
    batch_embeddings = embedding_model.encode(batch_lines, show_progress_bar=True)
    all_embeddings.append(batch_embeddings)

all_embeddings = np.vstack([b.astype(np.float32) for b in all_embeddings])
print(f"Shape of embeddings: {all_embeddings.shape}")

# 3. Initialise BERTopic with UMAP
umap_model = UMAP(n_neighbors=80, n_components=8, low_memory=True)

topic_model = BERTopic(
    embedding_model=None,
    umap_model=umap_model,
    min_topic_size=100,
    top_n_words=10,
    nr_topics=None,
    calculate_probabilities=True,
    verbose=True
)

# 4. Fit BERTopic
topics, probs = topic_model.fit_transform(
    sample_line_documents,
    embeddings=all_embeddings
)

print("\nBefore outlier reduction:")
print(f"Number of topics: {len(set(topics)) - (1 if -1 in set(topics) else 0)}")

Batches: 100%|██████████| 8/8 [00:00<00:00,  9.28it/s]


Shape of embeddings: (395773, 384)


2026-04-13 20:01:50,325 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-04-13 20:10:19,114 - BERTopic - Dimensionality - Completed ✓
2026-04-13 20:10:19,143 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-04-13 20:52:48,502 - BERTopic - Cluster - Completed ✓
2026-04-13 20:52:48,624 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-04-13 20:52:52,102 - BERTopic - Representation - Completed ✓



Before outlier reduction:
Number of topics: 297


After training, we have:
- `topic_model` → the trained BERTopic model
- `topics` → topic ID for each chunk (before outlier reduction)
- `probs` → confidence scores
- `all_embeddings` → vectors (expensive to recompute — keep these!)
- `sample_line_documents` → actual text chunks
- `song_map` → mapping chunk index → song index in sample_df

In [17]:
topics_array = np.array(topics)
print(f"Outliers before reduction: {(topics_array == -1).sum()}")

# 5. Reduce outliers
topics_reduced = topic_model.reduce_outliers(
    sample_line_documents,
    topics,
    strategy="c-tf-idf"
)

print("\nAfter outlier reduction:")
print(f"Number of topics: {len(set(topics_reduced)) - (1 if -1 in set(topics_reduced) else 0)}")
print(f"Outliers remaining: {(np.array(topics_reduced) == -1).sum()}")

# 6. Update topic assignments in model
topic_model.update_topics(sample_line_documents, topics=topics_reduced)

# 7. Final distribution
print("\nUpdated topic distribution (top 10):")
print(pd.Series(topics_reduced).value_counts().sort_index().head(10))

Outliers before reduction: 228655


2026-04-13 20:54:22,970 - BERTopic - WARNING: Using a custom list of topic assignments may lead to errors if topic reduction techniques are used afterwards. Make sure that manually assigning topics is the last step in the pipeline.Note that topic embeddings will also be created through weightedc-TF-IDF embeddings instead of centroid embeddings.



After outlier reduction:
Number of topics: 297
Outliers remaining: 1

Updated topic distribution (top 10):
-1        1
 0    13648
 1     9333
 2     5134
 3     4925
 4    11664
 5     3793
 6     6266
 7     4789
 8     3524
Name: count, dtype: int64


In [ ]:
line_df_v4 = pd.DataFrame({
    "song_idx": song_map,
    "topic": topics_reduced
})

In [19]:
topic_info = topic_model.get_topic_info()
print(topic_info.head(20))

topic_id = 0
print(topic_model.get_topic(topic_id))

    Topic  Count                             Name  \
0      -1      1                  -1_odule_bari__   
1       0  13648             0_she_her_woman_said   
2       1   9333         1_love_loving_loved_true   
3       2   5134           2_money_pay_cash_price   
4       3   4925        3_light_lights_shine_dark   
5       4  11664             4_go_back_gone_leave   
6       5   3793  5_dream_dreams_dreaming_dreamed   
7       6   6266             6_eyes_look_eye_face   
8       7   4789       7_night_wake_sleep_morning   
9       8   3524        8_fire_burn_burning_flame   
10      9   4008    9_heart_hearts_broken_beating   
11     10   3274          10_cold_summer_snow_ice   
12     11   4488             11_baby_good_love_oh   
13     12   2837        12_cry_tears_crying_cried   
14     13   2816       13_wine_drink_drunk_bottle   
15     14   3714      14_nigga_niggas_niggaz_fuck   
16     15   2691    15_sun_sunshine_sunny_sunrise   
17     16   3745         16_song_sing_music_so

In [ ]:
for i in range(0, 17):
    song = line_df_v4[line_df_v4['song_idx'] == i]
    if song.empty:
        print(f"Song {i}: No data\n")
        continue
    topic_counts = song['topic'].value_counts()
    dominant_topic = topic_counts.idxmax()
    print(f"Song {i}")
    print(topic_counts)
    print(f"Dominant topic: {dominant_topic}\n")

Song 0
topic
6      6
7      3
16     3
45     2
24     2
143    1
216    1
157    1
42     1
Name: count, dtype: int64
Dominant topic: 6

Song 1
topic
76     4
29     3
113    2
39     2
32     2
155    1
105    1
7      1
10     1
160    1
172    1
37     1
Name: count, dtype: int64
Dominant topic: 76

Song 2
topic
101    5
237    4
66     4
160    3
52     3
108    2
68     2
11     2
127    2
153    1
176    1
139    1
100    1
2      1
178    1
86     1
255    1
248    1
6      1
174    1
294    1
5      1
33     1
Name: count, dtype: int64
Dominant topic: 101

Song 3
topic
112    1
225    1
92     1
291    1
162    1
9      1
29     1
62     1
139    1
44     1
Name: count, dtype: int64
Dominant topic: 112

Song 4
topic
11     4
10     3
1      1
182    1
17     1
Name: count, dtype: int64
Dominant topic: 11

Song 5
topic
111    14
117    10
44      2
255     2
40      2
115     1
18      1
238     1
Name: count, dtype: int64
Dominant topic: 111

Song 6
topic
89     1
24     1
16

In [ ]:
# Flag songs where no topic appears more than once (potentially noisy)
song_chunks_grouped = line_df_v4.groupby('song_idx')
noisy_songs = song_chunks_grouped.filter(lambda x: x['topic'].value_counts().max() == 1)
print(f"Songs with no dominant topic: {noisy_songs['song_idx'].nunique()}")

Songs with no dominant topic: 1191


In [39]:
for i in range(len(topic_model.get_topics())):
    print(f'Topic {i}: {topic_model.get_topic(i)}')

Topic 0: [('she', 0.04507874363208452), ('her', 0.038046408967780554), ('woman', 0.006739790260177606), ('said', 0.0065425785621516875), ('girl', 0.0062562583938431645), ('was', 0.005656425426227531), ('is', 0.0043335966946454856), ('told', 0.004144187043351026), ('would', 0.003930105428674841), ('says', 0.00362675831822805)]
Topic 1: [('love', 0.03696240054460553), ('loving', 0.010062623036594765), ('loved', 0.009318375686494108), ('true', 0.009303867765258154), ('show', 0.006674894740622415), ('much', 0.006652889579386273), ('somebody', 0.005614806015571662), ('you', 0.00558488629944843), ('say', 0.005578792424898747), ('more', 0.005420475425544138)]
Topic 2: [('money', 0.05184344550416458), ('pay', 0.031630783245635674), ('cash', 0.014748110297272396), ('price', 0.01381806557686181), ('rich', 0.013281942510921654), ('paid', 0.01246290611230066), ('buy', 0.012107443061622735), ('dime', 0.011341258707175055), ('dollar', 0.010482021631234715), ('bank', 0.00901782559760184)]
Topic 3: [(

In [30]:
print(f"Topics distribution:\n{pd.Series(np.array(topics_reduced)).value_counts().sort_values(ascending=False).head(100)}")

Topics distribution:
0      13648
4      11664
1       9333
17      8629
74      8019
       ...  
202     1276
56      1275
95      1266
139     1264
248     1257
Name: count, Length: 100, dtype: int64


## Evaluation Metrics

In [36]:
# ── Coherence Score ──────────────────────────────────────────────────────────
tokenized_docs = [doc.split() for doc in sample_line_documents]
dictionary = Dictionary(tokenized_docs)

topics_dict = topic_model.get_topics()
topic_words = []
topic_ids_eval = []

for topic_id, words_weights in topics_dict.items():
    if topic_id != -1 and len(words_weights) >= 10:
        words = [word for word, _ in words_weights[:15]]
        topic_words.append(words)
        topic_ids_eval.append(topic_id)

coherence_model_cv = CoherenceModel(
    topics=topic_words,
    texts=tokenized_docs,
    dictionary=dictionary,
    coherence='c_v'
)
overall_coherence = coherence_model_cv.get_coherence()
print(f"Overall Coherence Score: {overall_coherence:.4f}")


# ── Topic Diversity ───────────────────────────────────────────────────────────
def compute_topic_diversity(topic_model, top_k=15):
    topics = topic_model.get_topics()
    all_words = []
    for topic_id, words in topics.items():
        if topic_id == -1:
            continue
        all_words.extend([word for word, _ in words[:top_k]])
    return len(set(all_words)) / len(all_words)

diversity_score = compute_topic_diversity(topic_model)
print(f"Topic Diversity Score: {diversity_score:.4f}")


Overall Coherence Score: 0.3633
Topic Diversity Score: 0.7293
